# Cross-Validation and Hyperparameter Tuning
Companion to the tuning notes.

## 1. How many random trials?

In [ ]:
import numpy as np
for p in [0.05, 0.01]:
    for target in [0.90, 0.95, 0.99]:
        print(f"top {p:.0%}, {target:.0%} chance: {int(np.ceil(np.log(1 - target) / np.log(1 - p)))} trials")
print("60 trials, top 5%:", round(1 - 0.95**60, 4))

## 2. Fold overlap

In [ ]:
for K in [3, 5, 10, 20]:
    print(K, f'{(K - 2) / (K - 1):.1%}')

## 3. Leakage: finding signal in pure noise

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 2000))        # 2000 columns of pure noise
y_noise = rng.integers(0, 2, size=200)        # a target that is pure noise too: the honest AUC is 0.5

# WRONG: choose the 20 "best" columns using all 200 rows, then cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=5, scoring="roc_auc")

# RIGHT: the selection is a pipeline step, so it is redone inside every fold on that fold's training part
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipe, X_noise, y_noise, cv=5, scoring="roc_auc")
print(f"selection outside CV: AUC {wrong.mean():.3f}     selection inside CV: AUC {right.mean():.3f}")

## 4. Random search for a random forest on the bank data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

In [ ]:
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score

param_distributions = {                       # no scaler: trees don't need one
    "n_estimators": randint(100, 400),
    "max_depth": [None, 6, 10, 16],
    "min_samples_leaf": randint(1, 30),
    "max_features": uniform(0.05, 0.5),       # a fraction of the 97 features, from 0.05 to 0.55
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_distributions,
                            n_iter=30, cv=cv, scoring="roc_auc", random_state=42)
search.fit(x_train, y_train)
print({k: (round(float(v), 3) if isinstance(v, float) else v) for k, v in search.best_params_.items()})
print(f"best CV AUC {search.best_score_:.4f}   test AUC {roc_auc_score(y_test, search.predict_proba(x_test)[:, 1]):.4f}")

In [ ]:
res = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(res[["mean_test_score", "std_test_score", "param_min_samples_leaf", "param_max_features"]].head(5).round(4).to_string(index=False))

## 5. Nested cross-validation

In [ ]:
from sklearn.model_selection import GridSearchCV
inner = GridSearchCV(make_pipeline(LogisticRegression(solver="liblinear", max_iter=2000)),
                     {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc")
inner.fit(x_train, y_train)
nested = cross_val_score(inner, x_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc")
print(f"best inner CV AUC (optimistic): {inner.best_score_:.4f}   nested CV AUC: {nested.mean():.4f} +/- {nested.std():.4f}")

## Try it yourself
1. Leakage example with k=200 and k=2.
2. Random search with 10 and 60 trials.
3. Describe datasets needing GroupKFold and TimeSeriesSplit.